In [1]:
import getpass
import os

def _set_if_undefined(var:str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'Provide your {var}')

_set_if_undefined('GROQ_API_KEY')
_set_if_undefined('TAVILY_API_KEY')

In [2]:
from typing import Annotated, List, Optional, Dict, Literal
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent   
from langchain_community.tools import TavilySearchResults
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from typing_extensions import TypedDict

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
@tool
def scrape_webpage(urls: List[str]) -> str:
    """User requests and bs4 to scrapes the provided web pages for deatiled information"""
    loader = WebBaseLoader(urls)
    docs = loader.load()
    return "\n\n".join(
        [
            f'<Document name="{doc.metadata.get("title", "")}">\n{doc.page_content}\n</Document>'  
        ]
    )

@tool   
def create_outline(
    points: Annotated[List[str], "List of main points or sections"],
    file_name: Annotated[str, "File path to save the outline"]
)-> Annotated[str, "Path of the saved outline file"]:
    """Create and save an outline"""
    file_to_use = os.path.join(os.getcwd(),'temp',file_name)
    with open(file_to_use, 'w') as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    
    return f'Outline saved to {file_name}'

@tool
def read_documents(
    file_name: Annotated[str, "File path to read the document form"],
    start: Annotated[Optional[int], "The start line.Default is 0"] = None,
    end: Annotated[Optional[int], "The end line.Default is None"] = None
):
    """Read the specified document"""
    file_to_use = os.path.join(os.getcwd(),'temp',file_name)
    with open(file_to_use, 'r') as file:
        lines = file.readlines()
    if start is None:
        start = 0
    return "\n".join(lines[start:end])

@tool
def write_documents(
    content: Annotated[str, "Text content to be written to the document"],
    file_name: Annotated[str, "File path to save the document"]
):
    """Create and save a text document"""
    file_to_use = os.path.join(os.getcwd(), 'temp', file_name)
    with open(file_to_use, 'w') as file:
        file.write(content)
    return f'Document saved to {file_name}'

@tool
def edit_document(
    file_name: Annotated[str, "File path to save the document"],
    insert: Annotated[Dict[int, str], "Dictionary where key is the line number and value is the text to be inserted at the line number"]
):
    """Edit a document by inserting text at specified line numbers"""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, 'r') as file:
        lines = file.readlines()
    sorted_insert = sorted(insert.items())
    
    for line_number, text in sorted_insert:
        if line_number <= len(lines):
            lines.insert(line_number, text + "\n")
        else:
            return f'Error: Line number {line_number} is out of bounds'
    
    with open(file_to_use, 'w') as file:
        file.writelines(lines)
    
    return f'Document {file_name} edited and saved successfully'

In [4]:
@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart"],
):
    """Use this to execute python code. If you want to see the output of any value,
    you should print it with `print(...)`. This is visible to the user"""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed: \n ```python \n{code}\n``` \n Stdout: {result}"

In [5]:
class State(MessagesState):
    next: str

def make_supervisor_node(llm: BaseChatModel, members: List[str]) -> str:
    options = ["FINISH"] + members
    system_prompt = (
        "You are a supervisor tasked with managing a conversation between the"
        f" following workers: {members}. Given the following user request,"
        " respond with the worker to act next. Each worker will perform a"
        " task and respond with their results and status. When finished,"
        " respond with FINISH."
    )


    class Router(TypedDict):
        next: Literal[*options]
    
    def supervisor_node(state: State) -> Command[Literal[*members, "__end__"]]:
        messages = [
            {"role": "system", "content": system_prompt}
        ] + state["messages"]
    
        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        if goto == "FINISH":
            goto = END
        return Command(goto=goto, update={"next": goto})
    
    return supervisor_node

In [9]:
# llama-3.3-70b-versatile supports both tool calling AND structured output
# No need for a separate tool_llm — one model handles everything
llm = ChatGroq(model='llama-3.3-70b-versatile')

tavily_tool = TavilySearchResults(max_results=3)

search_agent = create_react_agent(llm, tools=[tavily_tool])

def search_node(state: State) -> Command[Literal['supervisor']]:
    result = search_agent.invoke(state)
    return Command(
        update={
            'messages': [HumanMessage(content=result['messages'][-1].content, name='search')]
        },
        goto='supervisor'
    )

web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpage])

def web_scrapper_node(state: State) -> Command[Literal['supervisor']]:
    result = web_scrapper_agent.invoke(state)
    return Command(
        update={
            'messages': [HumanMessage(content=result['messages'][-1].content, name='web_scrapper')]
        },
        goto='supervisor'
    )

research_supervisor_node = make_supervisor_node(llm, ['search', 'web_scrapper'])


C:\Users\DINESHWAR\AppData\Local\Temp\ipykernel_6292\3497752206.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  search_agent = create_react_agent(llm, tools=[tavily_tool])
C:\Users\DINESHWAR\AppData\Local\Temp\ipykernel_6292\3497752206.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpage])


In [10]:
research_builder = StateGraph(State)
research_builder.add_node("supervisor", research_supervisor_node)
research_builder.add_node("search", search_node)
research_builder.add_node("web_scrapper", web_scrapper_node)

research_builder.add_edge(START, "supervisor")
research_graph = research_builder.compile()

In [12]:
for s in research_graph.stream(
    {"messages" : [("user", "what is the weather like in Porto today?")]},
    {"recursion_limit": 20},
): 
    print(s)
    print("---")

{'supervisor': {'next': 'search'}}
---
{'search': {'messages': [HumanMessage(content='The current weather in Porto is mostly cloudy with a high of 67°F (19°C) and a low of 53°F (12°C). There is a chance of showers late in the morning. The air quality is generally acceptable for most individuals, but sensitive groups may experience minor to moderate symptoms from long-term exposure.', additional_kwargs={}, response_metadata={}, name='search', id='e1e4f353-e688-4081-a419-ebace4c41c09')]}}
---
{'supervisor': {'next': '__end__'}}
---


In [13]:
doc_writer_agent = create_react_agent(
    llm, 
    tools = [write_documents, edit_document, read_documents],
    prompt = (
        "You can read, write and edit documents based on note taker's outlines. "
        "Don't ask follow up questions."
    )
)

def doc_writing_node(state: State) -> Command[Literal["supervisor"]]:
    result = doc_writer_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "doc_writer")
            ]
        },
        goto = "supervisor",
    )

note_taking_agent = create_react_agent(
    llm,
    tools = [create_outline, read_documents],
    prompt = (
        "You can read documents and create outlines for the document writer."
        "Don\'t ask follow up questions."
    )
)


def note_taking_node(state: State) -> Command[Literal["supervisor"]]:
    result = note_taking_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "note_taker")
            ]
        },
        goto = "supervisor",
    )

chart_generating_agent = create_react_agent(
    llm, tools = [read_documents, python_repl_tool]
)

def chart_generating_node(state: State) -> Command[Literal["supervisor"]]:
    result = chart_generating_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "chart_generator")
            ]
        },
        goto = "supervisor",
    )

doc_writing_supervisor_node = make_supervisor_node(
    llm, ["doc_writer", "note_taker", "chart_generator"]
)

C:\Users\DINESHWAR\AppData\Local\Temp\ipykernel_6292\752080632.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  doc_writer_agent = create_react_agent(
C:\Users\DINESHWAR\AppData\Local\Temp\ipykernel_6292\752080632.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  note_taking_agent = create_react_agent(
C:\Users\DINESHWAR\AppData\Local\Temp\ipykernel_6292\752080632.py:42: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  chart_generating_agent = create_react_agent(


In [14]:
writing_builder = StateGraph(State)
writing_builder.add_node("supervisor", doc_writing_supervisor_node)
writing_builder.add_node("doc_writer", doc_writing_node)
writing_builder.add_node("note_taker", note_taking_node)
writing_builder.add_node("chart_generator", chart_generating_node)

writing_builder.add_edge(START, "supervisor")
writing_graph = writing_builder.compile()

In [17]:
for s in writing_graph.stream(
    {
        "messages": [
            (
                "user",
                "Write an outline for a poem about dogs and after that write the poem itself and store it"
            )
        ]
    },
    {"recursion_limit": 30}
):
    print(s)
    print("---")

{'supervisor': {'next': 'note_taker'}}
---


BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool read_documents did not match schema: errors: [`/start`: expected integer, but got string, `/start`: expected null, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=read_documents>{"file_name": "final_dogs_poem", "start": "0"}</function>'}}